# 03 · Exploratory analysis

What goes with a higher market value? One question per chart, using the player-season table from notebook 02.

Market value is skewed, so charts use a **log scale** for value. Performance charts only use rows with Premier League data (seasons spent in other leagues have blank performance stats, so they simply don't appear); per-90 charts also need at least 450 minutes. Age, position and club charts use all rows.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src import config, plots

sns.set_theme(style='whitegrid')
df = pd.read_csv(config.PROCESSED_DIR / 'player_season.csv', parse_dates=['value_date'])
latest = df[df['season'] == config.CURRENT_SEASON]
perf = df[df['pl_minutes'] >= config.MIN_MINUTES_PER90]  # rows with per-90 stats
print(len(df), 'player-seasons |', len(perf), 'with per-90 stats |', len(latest), 'in the latest season')

millions = mtick.FuncFormatter(lambda x, _: f'{x/1e6:g}M')


def save(name):
    config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(config.FIGURES_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

## 1. How does value change with age?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=df, x='age', y='market_value_in_eur', alpha=0.25, s=18, ax=ax)
median_by_age = df.groupby(df['age'].round())['market_value_in_eur'].median()
ax.plot(median_by_age.index, median_by_age.values, color='crimson', lw=2, label='median by age')
ax.set_yscale('log'); ax.yaxis.set_major_formatter(millions)
ax.set(xlabel='Age', ylabel='Market value (EUR, log scale)', title='Value vs age'); ax.legend()
save('value_vs_age')

## 2. Does position matter?

In [ ]:
plots.median_value_bar(df, 'position', 'Median value by position', figsize=(8, 3.5))
save('value_by_position')

In [ ]:
plots.median_value_bar(df, 'sub_position', 'Median value by sub-position', figsize=(8, 5.5))
save('value_by_sub_position')

Each bar is the median player-season value for that group (half the players are worth more, half less); `n` is how many player-seasons it is based on. Groups with a small `n` (like the midfield wing roles) are not reliable.

## 3. Does the club matter? (latest season)

Grouped by each player's *current* Premier League club. The club on a valuation record is where the player was at that moment (loans, U21 sides, other leagues), so it is not used here.

In [ ]:
club_median = latest.groupby('current_club')['market_value_in_eur'].agg(['median', 'size']).sort_values('median')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(club_median.index, club_median['median'])
ax.xaxis.set_major_formatter(millions)
ax.set(xlabel='Median market value (EUR)', title=f'Median player value by club, {config.CURRENT_SEASON}/{(config.CURRENT_SEASON + 1) % 100}')
save('value_by_club')

## 4. Does playing time matter?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=df, x='pl_minutes', y='market_value_in_eur', hue='position', alpha=0.5, s=20, ax=ax)
ax.set_yscale('log'); ax.yaxis.set_major_formatter(millions)
ax.set(xlabel='Premier League minutes in the season', ylabel='Market value (EUR, log scale)', title='Value vs minutes played')
save('value_vs_minutes')

## 5. Do goals and assists matter?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, col in zip(axes, ['goals_per90', 'assists_per90']):
    sns.scatterplot(data=perf, x=col, y='market_value_in_eur', hue='position', alpha=0.5, s=20, ax=ax, legend=(col == 'assists_per90'))
    ax.set_yscale('log'); ax.yaxis.set_major_formatter(millions)
    ax.set(xlabel=col.replace('_', ' '), ylabel='Market value (EUR, log scale)')
axes[0].set_title('Goals per 90'); axes[1].set_title('Assists per 90')
plt.tight_layout(); save('value_vs_goals_assists')

Attackers should show the clearest link. Defenders and goalkeepers hardly score, so goals per 90 says little about their value.

## 6. Does the previous season's value predict this season's?

In [ ]:
mom = df.dropna(subset=['prev_value'])
fig, ax = plt.subplots(figsize=(5.5, 5.5))
sns.scatterplot(data=mom, x='prev_value', y='market_value_in_eur', alpha=0.3, s=18, ax=ax)
ax.set_xscale('log'); ax.set_yscale('log')
ax.xaxis.set_major_formatter(millions); ax.yaxis.set_major_formatter(millions)
ax.set(xlabel='Previous season value (EUR)', ylabel='Value (EUR)', title='Value vs previous season value')
save('value_vs_prev_value')
print('correlation of log values:', round(np.corrcoef(np.log1p(mom['prev_value']), mom['log_value'])[0, 1], 3))

Value is very sticky from one season to the next, so any model that includes last season's value will look good for a boring reason. We should keep this in mind when we get to modelling.

## 7. Height and foot

In [ ]:
plots.median_value_bar(df, 'foot', 'Median value by preferred foot', figsize=(8, 2.8))
save('value_by_foot')

In [ ]:
height_bands = pd.cut(df['height_in_cm'], [0, 174, 179, 184, 189, 250], labels=['under 175 cm', '175-179 cm', '180-184 cm', '185-189 cm', '190 cm and over'])
plots.median_value_bar(df.assign(height_band=height_bands), 'height_band', 'Median value by height', figsize=(8, 3.5), sort_by_value=False)
save('value_by_height')

## 8. Correlation overview

In [ ]:
cols = ['log_value', 'age', 'height_in_cm', 'pl_apps', 'pl_starts', 'pl_minutes', 'pl_goals', 'pl_assists', 'goals_per90', 'assists_per90', 'pl_yellow']
corr = df[cols].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax)
ax.set_title('Correlation between features and log value')
save('correlation_heatmap')

In [ ]:
corr['log_value'].drop('log_value').sort_values(ascending=False).round(2)

Correlation measures straight-line association only, and says nothing about cause. Many of these features overlap (minutes, starts and appearances move together).

## 9. Findings so far

_To be filled in after we look at the charts together._